In [ ]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance

In [ ]:
# Load dataset
df = pd.read_excel("project_dataset.xlsx")

In [ ]:
# Separate features, target, and country names
X = df.drop(columns=["Country", "BiodiversityIndex"])
y = df["BiodiversityIndex"]
countries = df["Country"]

In [ ]:
# Split dataset
X_train, X_test, y_train, y_test, countries_train, countries_test = train_test_split(
    X, y, countries, test_size=0.2, random_state=42
)

In [ ]:
# GridSearchCV for optimal k
pipeline = make_pipeline(StandardScaler(), KNeighborsRegressor())
param_grid = {'kneighborsregressor__n_neighbors': range(1, 21)}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_k = grid_search.best_params_['kneighborsregressor__n_neighbors']
print(f"Best K found: {best_k}")


In [ ]:
# Prediction
X_test_scaled = pd.DataFrame(
    best_model.named_steps["standardscaler"].transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)
y_pred = best_model.predict(X_test_scaled)
y_pred = np.clip(y_pred, 0, 1)

In [ ]:
# Evaluation
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)


print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

In [ ]:
# Plot: Actual vs Predicted
x = np.arange(len(countries_test))
height = 0.35

plt.figure(figsize=(10, 16))
plt.barh(x - height/2, y_test, height, label='Actual', color='seagreen', alpha=0.8)
plt.barh(x + height/2, y_pred, height, label='Predicted', color='skyblue', alpha=0.8)

for i, v in enumerate(y_test):
    plt.text(v + 0.01, i - height/2, f"{v:.4f}", va='center', fontsize=8)
for i, v in enumerate(y_pred):
    plt.text(v + 0.01, i + height/2, f"{v:.4f}", va='center', fontsize=8)

plt.yticks(x, countries_test, fontsize=8)
plt.xlabel("Biodiversity Index")
plt.title("Actual vs Predicted Biodiversity Index (KNN)")
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Absolute error
error = np.abs(y_test - y_pred)

plt.figure(figsize=(18, 7))
plt.bar(countries_test, error, color="salmon")

for i, v in enumerate(error):
    plt.text(i, v + 0.005, f"{v:.4f}", color='black', ha='center', fontsize=8)

plt.xticks(rotation=90)
plt.ylabel("Absolute Error")
plt.xlabel("Country")
plt.title("Prediction Error (Absolute Error, Test Countries) - KNN")
plt.tight_layout()
plt.show()


In [ ]:
result = permutation_importance(best_model, X_test_scaled, y_test, scoring='r2', n_repeats=10, random_state=42)
importances = result.importances_mean
indices = np.argsort(importances)[::-1]
features_sorted = X.columns[indices]

plt.figure(figsize=(15, 12))
plt.barh(features_sorted, importances[indices], color="steelblue")
for i, v in enumerate(importances[indices]):
    plt.text(v + 0.001, i, f"{v:.5f}", color='black', va='center', fontsize=8)

plt.title("Feature Importance - KNN (Permutation Importance)")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
# Group features into categories and calculate summed importance for each
economy_features = ['GDP_PPP', 'HDI', 'Surfacearea']
natural_features = ['Latitude', 'Forestarea(%)', 'Waterresources(%)']
climate_features = [f for f in X.columns if f not in economy_features + natural_features]

category_scores = {
    'Economy': sum(importances[X.columns.get_loc(f)] for f in economy_features),
    'Natural Environment': sum(importances[X.columns.get_loc(f)] for f in natural_features),
    'Climate': sum(importances[X.columns.get_loc(f)] for f in climate_features),
}

df_grouped = pd.DataFrame.from_dict(category_scores, orient='index', columns=['Importance'])

# Pie chart
labels = df_grouped.index
sizes = df_grouped['Importance']
colors = ['#66c2a5', '#fc8d62', '#8da0cb']

plt.figure(figsize=(6, 6))
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140)
plt.title("Grouped Feature Importance by Category (KNN)")
plt.axis('equal')
plt.tight_layout()
plt.show()
